# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratheekrao12/-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

This notebook uses the March 1–15 analytical feature vector from Week 5: `early_impressions`, `early_clicks`, `weighted_avg_position`, `active_search_days`, and `early_ctr`. One row represents a pseudonymized `client_hash_id × content_hash_id` page pair. Identifiers are grouping context only. The proxy target is computed separately from March 16–31 data.

In [1]:
from pathlib import Path
import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

token = os.getenv('HF_TOKEN')
if not token:
    env_path = next((base/'.env' for base in [Path.cwd(), *Path.cwd().parents] if (base/'.env').exists()), None)
    if env_path:
        for line in env_path.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                token = line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))

march_url = hf_hub_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset', filename='fact_content_daily_performance/month=2026-03/data_0.parquet', token=token)
con = duckdb.connect()
feature_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position*gsc_impressions ELSE 0 END)
      / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END), 0) AS weighted_avg_position,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS active_search_days,
    SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS late_impressions,
    SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_clicks,
    COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS late_available_days
  FROM read_parquet(?)
  WHERE gsc_data_available IS TRUE
  GROUP BY client_hash_id, content_hash_id
), feature_frame AS (
  SELECT *,
    CASE WHEN early_impressions > 0 THEN 100.0*early_clicks/early_impressions ELSE NULL END AS early_ctr,
    CASE WHEN late_available_days = 16 AND late_impressions/16.0 < early_impressions/15.0 THEN 1 ELSE 0 END AS is_declining_proxy
  FROM page_windows
)
SELECT * FROM feature_frame
WHERE early_impressions > 0 AND weighted_avg_position IS NOT NULL AND active_search_days > 0
ORDER BY hash(client_hash_id || content_hash_id)
LIMIT 30000
'''
df = con.execute(feature_query, [march_url]).fetchdf()
feature_columns = ['early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr']
feature_vector = df[['client_hash_id','content_hash_id'] + feature_columns].copy()
print('March feature-vector rows:', len(feature_vector))
print('Unique client/page pairs:', feature_vector[['client_hash_id','content_hash_id']].drop_duplicates().shape[0])
print('Predictive feature columns:', feature_columns)
print(feature_vector.head(8)[['client_hash_id','content_hash_id'] + feature_columns].to_string(index=False))
assert len(feature_vector) == feature_vector[['client_hash_id','content_hash_id']].drop_duplicates().shape[0]
assert set(feature_columns).issubset(feature_vector.columns)
assert 'is_declining_proxy' not in feature_columns

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


March feature-vector rows: 30000
Unique client/page pairs: 30000
Predictive feature columns: ['early_impressions', 'early_clicks', 'weighted_avg_position', 'active_search_days', 'early_ctr']
         client_hash_id          content_hash_id  early_impressions  early_clicks  weighted_avg_position  active_search_days  early_ctr
client_23a62021009f63c4 content_f5785c484503a6e1              154.0           0.0              19.097403                  15        0.0
client_62f4a7e64f5e0096 content_378cab59bb132b3b               21.0           0.0              72.666667                  11        0.0
client_a80fca3f171ed1de content_d216247d6d8f6d39                7.0           0.0               4.000000                   6        0.0
client_3f0ce4d44fe94f3d content_04519ccc2cf78e25              298.0           0.0              41.536913                  15        0.0
client_3ffa76342f366962 content_7899ed5075ab5665                3.0           0.0               1.333333                   2     

## 2. Feature notes (meaning, missing, categorical, available-when?)

All five fields use March 1–15. There are no categorical model inputs. Missing values are checked below and no imputation silently invents data. `early_ctr` is derived only from the early-window numerator and denominator.

In [2]:
feature_notes = pd.DataFrame([
    {'Feature':'early_impressions','Meaning':'GSC impressions, March 1-15','Window':'2026-03-01 to 2026-03-15','Missing_count':int(df['early_impressions'].isna().sum()),'Available_at_prediction':'Yes; RETAIN'},
    {'Feature':'early_clicks','Meaning':'GSC clicks, March 1-15','Window':'2026-03-01 to 2026-03-15','Missing_count':int(df['early_clicks'].isna().sum()),'Available_at_prediction':'Yes; RETAIN'},
    {'Feature':'weighted_avg_position','Meaning':'Impression-weighted average position, March 1-15','Window':'2026-03-01 to 2026-03-15','Missing_count':int(df['weighted_avg_position'].isna().sum()),'Available_at_prediction':'Yes; RETAIN'},
    {'Feature':'active_search_days','Meaning':'Distinct available GSC dates, March 1-15','Window':'2026-03-01 to 2026-03-15','Missing_count':int(df['active_search_days'].isna().sum()),'Available_at_prediction':'Yes; RETAIN'},
    {'Feature':'early_ctr','Meaning':'100 x early clicks / early impressions','Window':'2026-03-01 to 2026-03-15','Missing_count':int(df['early_ctr'].isna().sum()),'Available_at_prediction':'Yes; RETAIN'},
])
print(feature_notes.to_string(index=False))
print('Feature missingness:')
print(feature_vector[feature_columns].isna().sum().to_string())
print('Complete predictive rows:', int(feature_vector[feature_columns].notna().all(axis=1).sum()))
print('Target-evaluable 16-day later rows:', int((df['late_available_days']==16).sum()))
assert feature_vector[feature_columns].notna().all().all()

              Feature                                          Meaning                   Window  Missing_count Available_at_prediction
    early_impressions                      GSC impressions, March 1-15 2026-03-01 to 2026-03-15              0             Yes; RETAIN
         early_clicks                           GSC clicks, March 1-15 2026-03-01 to 2026-03-15              0             Yes; RETAIN
weighted_avg_position Impression-weighted average position, March 1-15 2026-03-01 to 2026-03-15              0             Yes; RETAIN
   active_search_days         Distinct available GSC dates, March 1-15 2026-03-01 to 2026-03-15              0             Yes; RETAIN
            early_ctr           100 x early clicks / early impressions 2026-03-01 to 2026-03-15              0             Yes; RETAIN
Feature missingness:
early_impressions        0
early_clicks             0
weighted_avg_position    0
active_search_days       0
early_ctr                0
Complete predictive rows: 30000
Ta

## 3. The leakage hunt

Below is a deliberately negative experiment. The honest client-grouped model uses only the five March 1–15 features.

The comparison model additionally receives `late_impressions`, `late_clicks`, and `is_declining_proxy`. `is_declining_proxy` is direct target leakage: its value is the answer the model is being asked to estimate. The resulting score is an intentionally invalid upper-bound demonstration, not a realistic predictive model, and it must not be interpreted as a valid model comparison or evidence of improvement.

`late_impressions` and `late_clicks` are future-window leakage. The contrast shows how leakage can inflate apparent performance; neither leaky feature set nor its result is used as the approved model.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

evaluable = df[df['late_available_days'].eq(16)].copy()
X = evaluable[feature_columns].copy()
y = evaluable['is_declining_proxy'].astype(int)
groups = evaluable['client_hash_id']
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(X, y, groups))

def auc_for(columns):
    frame = evaluable[list(columns)].copy()
    split_model = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])
    split_model.fit(frame.iloc[train_idx], y.iloc[train_idx])
    return float(roc_auc_score(y.iloc[test_idx], split_model.predict_proba(frame.iloc[test_idx])[:, 1]))

honest_auc = auc_for(feature_columns)
leaky_columns = feature_columns + ['late_impressions','late_clicks','is_declining_proxy']
leaky_auc = auc_for(leaky_columns)
train_overlap = len(set(evaluable.iloc[train_idx]['client_hash_id']) & set(evaluable.iloc[test_idx]['client_hash_id']))
print('Honest client-grouped ROC-AUC:', round(honest_auc, 4))
print('Intentionally invalid leaky/direct-target ROC-AUC upper bound:', round(leaky_auc, 4))
print('Direct target leakage: is_declining_proxy is the label, not a predictive feature.')
print('This upper bound is not a valid model comparison or predictive result.')
print('Score change with future/target fields:', round(leaky_auc - honest_auc, 4))
print('Evaluable target rows:', len(evaluable))
print('Train/test client overlap:', train_overlap)
assert train_overlap == 0
assert not {'late_impressions','late_clicks','is_declining_proxy'} & set(feature_columns)

Honest client-grouped ROC-AUC: 0.7195
Intentionally invalid leaky/direct-target ROC-AUC upper bound: 1.0
Direct target leakage: is_declining_proxy is the label, not a predictive feature.
This upper bound is not a valid model comparison or predictive result.
Score change with future/target fields: 0.2805
Evaluable target rows: 16846
Train/test client overlap: 0


## 4. What I excluded and why

The final feature vector excludes late traffic totals, the target/proxy, target siblings, product/action fields, URLs/query context, and identifiers as predictors. Identifiers remain allowed for joins and client-grouped validation.

In [4]:
excluded_fields = pd.DataFrame([
    {'Field':'late_impressions / late_clicks','Reason':'March 16-31 future window; unavailable at prediction time and directly overlapping target construction.'},
    {'Field':'is_declining_proxy','Reason':'The label itself; target leakage makes training trivially answer the answer column.'},
    {'Field':'trend_pct / trend_direction','Reason':'Target-derived or movement sibling fields; excluded from predictive features.'},
    {'Field':'client_hash_id / content_hash_id','Reason':'Safe only for joins/grouping; ID memorization is not a durable predictive signal.'},
    {'Field':'product flags / action labels','Reason':'Workflow-derived fields may encode prior decisions and create circular evaluation.'},
    {'Field':'URLs, names, titles, raw queries','Reason':'Private or unnecessary context; excluded for public-safe analysis.'},
])
print(excluded_fields.to_string(index=False))
print('Five final features:', ', '.join(feature_columns))
print('Leakage audit complete:', True)
assert set(feature_columns) == {'early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr'}
assert feature_vector[['client_hash_id','content_hash_id']].notna().all().all()

                           Field                                                                                                  Reason
  late_impressions / late_clicks March 16-31 future window; unavailable at prediction time and directly overlapping target construction.
              is_declining_proxy                     The label itself; target leakage makes training trivially answer the answer column.
     trend_pct / trend_direction                           Target-derived or movement sibling fields; excluded from predictive features.
client_hash_id / content_hash_id                       Safe only for joins/grouping; ID memorization is not a durable predictive signal.
   product flags / action labels                      Workflow-derived fields may encode prior decisions and create circular evaluation.
URLs, names, titles, raw queries                                      Private or unnecessary context; excluded for public-safe analysis.
Five final features: early_impressions, e

## Self-check

- [x] Feature vector uses only five March 1–15 predictors.
- [x] Feature meanings and availability are documented.
- [x] Missing values and page-ID grain are reported.
- [x] Future and target-derived fields are identified and tested only as an explicitly leaky contrast.
- [x] Client identifiers are excluded as model inputs and retained only for grouped evaluation.
- [x] Product flags, IDs-as-features, private names/URLs/queries, and future-window fields are excluded.
- [x] Honest-versus-leaked evidence is generated from the available target-evaluable rows.
- [x] All conclusions remain observed, directional, and public-safe.